In [6]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
#!/usr/bin/env python3
"""
Contrastive training of SoundVector track embeddings.
"""

import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.insert(0, '/content/')
from model import FeatureBank, TrackEncoder


class GroupSampler:
    """Samples (anchor, positive) index pairs uniformly within groups of size >= 2."""

    def __init__(self, gids: np.ndarray):
        valid = gids >= 0
        order = np.argsort(gids[valid], kind="stable")
        self.sorted_idx = np.flatnonzero(valid)[order]          # track ids sorted by gid
        sorted_gids = gids[self.sorted_idx]
        unique, self.group_ptr, counts = np.unique(
            sorted_gids, return_index=True, return_counts=True)
        self.group_sizes = counts

        # Eligible anchors: members of groups with >= 2 tracks
        group_pos = np.searchsorted(unique, sorted_gids)         # group slot per sorted row
        eligible = counts[group_pos] >= 2
        self.anchor_rows = np.flatnonzero(eligible)              # rows into sorted_idx
        self.anchor_group = group_pos[eligible]
        self.anchor_rank = self.anchor_rows - self.group_ptr[self.anchor_group]
        self.n_eligible = len(self.anchor_rows)

    def sample(self, n: int, rng: np.random.Generator):
        pick = rng.integers(0, self.n_eligible, size=n)
        rows = self.anchor_rows[pick]
        group = self.anchor_group[pick]
        size = self.group_sizes[group]
        slot = rng.integers(0, size - 1)
        slot += slot >= self.anchor_rank[pick]                   # skip the anchor itself
        anchors = self.sorted_idx[rows]
        positives = self.sorted_idx[self.group_ptr[group] + slot]
        return anchors, positives


def pick_device(name: str) -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def info_nce(z_a, z_p, temperature: float):
    logits = z_a @ z_p.T / temperature
    labels = torch.arange(len(z_a), device=z_a.device)
    loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
    acc = (logits.argmax(dim=1) == labels).float().mean()
    return loss, acc


def encode_catalog(model, bank, device, batch_size=16384):
    model.eval()
    out = np.zeros((bank.n_tracks, model.embed_dim), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, bank.n_tracks, batch_size):
            idx = np.arange(start, min(start + batch_size, bank.n_tracks))
            x = torch.from_numpy(bank.batch(idx)).to(device)
            out[idx] = model(x).cpu().numpy()
    return out

In [2]:
def main(args_list=None):
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--artifacts", default="artifacts")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--pairs-per-epoch", type=int, default=4_000_000)
    ap.add_argument("--batch-size", type=int, default=2048)
    ap.add_argument("--embed-dim", type=int, default=128)
    ap.add_argument("--hidden", type=int, nargs="+", default=[1024, 512])
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--weight-decay", type=float, default=1e-4)
    ap.add_argument("--temperature", type=float, default=0.1)
    ap.add_argument("--genre-dropout", type=float, default=0.3)
    ap.add_argument("--source-weights", type=float, nargs=4, default=[0.45, 0.25, 0.20, 0.10])
    ap.add_argument("--device", default="auto")
    ap.add_argument("--seed", type=int, default=42)

    # Pass args_list to parse_args so it works in Notebooks
    args = ap.parse_args(args_list)


    device = pick_device(args.device)
    rng = np.random.default_rng(args.seed)
    torch.manual_seed(args.seed)
    print(f"Device: {device}")

    bank = FeatureBank(args.artifacts)
    meta = pd.read_parquet(os.path.join(args.artifacts, "meta.parquet"))
    print(f"{bank.n_tracks:,} tracks | input dim {bank.input_dim} "
          f"({bank.dense_dim} dense + {bank.genre_vocab_size} genres)")

    sources, weights = [], []
    for name, w in zip(["artist_gid", "album_gid", "genreset_gid", "chart_gid"],
                       args.source_weights):
        sampler = GroupSampler(meta[name].to_numpy())
        if sampler.n_eligible and w > 0:
            sources.append((name, sampler))
            weights.append(w)
            print(f"  pair source {name:13s} weight {w:.2f} "
                  f"({sampler.n_eligible:,} eligible anchors)")
    weights = np.array(weights) / np.sum(weights)

    model = TrackEncoder(bank.dense_dim, bank.genre_vocab_size,
                         tuple(args.hidden), args.embed_dim).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Encoder: {n_params/1e6:.1f}M params, embed dim {args.embed_dim}")

    # Optimization: Compile the model for A100 (PyTorch 2.0+)
    if device.type == "cuda" and hasattr(torch, "compile"):
        print("Compiling model for faster execution...")
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                  weight_decay=args.weight_decay)
    steps_per_epoch = args.pairs_per_epoch // args.batch_size
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=args.epochs * steps_per_epoch)

    # Optimization: Automatic Mixed Precision (AMP)
    scaler = torch.amp.GradScaler(device='cuda') if device.type == "cuda" else None

    log = {"args": vars(args) | {"device": str(device)}, "epochs": []}
    for epoch in range(1, args.epochs + 1):
        # Pre-sample the epoch's pairs (vectorized), then shuffle across sources
        counts = rng.multinomial(args.pairs_per_epoch, weights)
        anchors, positives = [], []
        for (name, sampler), c in zip(sources, counts):
            a, p = sampler.sample(c, rng)
            anchors.append(a)
            positives.append(p)
        anchors = np.concatenate(anchors)
        positives = np.concatenate(positives)
        perm = rng.permutation(len(anchors))
        anchors, positives = anchors[perm], positives[perm]

        model.train()
        t0, losses, accs = time.time(), [], []
        for step in range(steps_per_epoch):
            lo, hi = step * args.batch_size, (step + 1) * args.batch_size
            batch_idx = np.concatenate([anchors[lo:hi], positives[lo:hi]])
            x = bank.batch(batch_idx)
            # Genre-block dropout, independent per sample
            drop = rng.random(len(x)) < args.genre_dropout
            x[drop, bank.dense_dim:] = 0.0

            # Optimization: Pin memory and use non_blocking transfer
            x = torch.from_numpy(x)
            if device.type == "cuda":
                x = x.pin_memory().to(device, non_blocking=True)
            else:
                x = x.to(device)

            optimizer.zero_grad(set_to_none=True)

            # Optimization: Mixed precision forward and backward passes
            if scaler is not None:
                with torch.amp.autocast(device_type='cuda'):
                    z = model(x)
                    z_a, z_p = z[:len(z) // 2], z[len(z) // 2:]
                    loss, acc = info_nce(z_a, z_p, args.temperature)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                z = model(x)
                z_a, z_p = z[:len(z) // 2], z[len(z) // 2:]
                loss, acc = info_nce(z_a, z_p, args.temperature)
                loss.backward()
                optimizer.step()

            scheduler.step()
            losses.append(loss.item())
            accs.append(acc.item())

            if (step + 1) % 200 == 0:
                rate = (step + 1) * args.batch_size / (time.time() - t0)
                print(f"  epoch {epoch} step {step+1}/{steps_per_epoch} "
                      f"loss {np.mean(losses[-200:]):.4f} "
                      f"in-batch acc {np.mean(accs[-200:]):.3f} "
                      f"({rate:,.0f} pairs/s)", flush=True)

        epoch_stats = {"epoch": epoch, "loss": float(np.mean(losses)),
                       "in_batch_acc": float(np.mean(accs)),
                       "seconds": round(time.time() - t0, 1)}
        log["epochs"].append(epoch_stats)
        print(f"Epoch {epoch}: loss {epoch_stats['loss']:.4f} "
              f"acc {epoch_stats['in_batch_acc']:.3f} ({epoch_stats['seconds']}s)")

    # If the model was compiled, save the original module's state dict
    model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model
    model_path = os.path.join(args.artifacts, "encoder.pt")
    model_to_save.save(model_path)
    print(f"Saved {model_path}")

    print("Encoding full catalog...")
    embeddings = encode_catalog(model_to_save, bank, device)
    emb_path = os.path.join(args.artifacts, "embeddings.npy")
    np.save(emb_path, embeddings)
    print(f"Saved {emb_path} {embeddings.shape}")

    with open(os.path.join(args.artifacts, "train_log.json"), "w") as f:
        json.dump(log, f, indent=2)


In [3]:
if __name__ == "__main__":
    import sys

    if "ipykernel" in sys.modules:
        main([
            "--artifacts", "/content/drive/MyDrive/soundVector/artifacts/",
            "--device", "cuda",
            "--batch-size", "32768",
            "--pairs-per-epoch", "10000000",
            "--epochs", "30",
            "--embed-dim", "256", #512
            "--hidden", "2048", "1024", "512" # 4096
            # "--lr", "4e-3"
        ])
    else:
        pass

Device: cuda
899,224 tracks | input dim 2028 (28 dense + 2000 genres)
  pair source artist_gid    weight 0.45 (833,609 eligible anchors)
  pair source album_gid     weight 0.25 (530,530 eligible anchors)
  pair source genreset_gid  weight 0.20 (682,820 eligible anchors)
  pair source chart_gid     weight 0.10 (1,521 eligible anchors)
Encoder: 6.9M params, embed dim 256
Compiling model for faster execution...
  epoch 1 step 200/305 loss 5.5576 in-batch acc 0.189 (76,718 pairs/s)
Epoch 1: loss 5.3174 acc 0.201 (126.0s)
  epoch 2 step 200/305 loss 4.6376 in-batch acc 0.239 (85,318 pairs/s)
Epoch 2: loss 4.5860 acc 0.243 (117.2s)
  epoch 3 step 200/305 loss 4.3834 in-batch acc 0.258 (85,876 pairs/s)
Epoch 3: loss 4.3543 acc 0.261 (116.6s)
  epoch 4 step 200/305 loss 4.2373 in-batch acc 0.271 (85,410 pairs/s)
Epoch 4: loss 4.2187 acc 0.272 (117.2s)
  epoch 5 step 200/305 loss 4.1387 in-batch acc 0.279 (85,232 pairs/s)
Epoch 5: loss 4.1250 acc 0.281 (117.3s)
  epoch 6 step 200/305 loss 4.061

In [5]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp312-cp312-linux_x86_64.whl size=2733619 sha256=793d427ab5984b8c9fdd0d5bfc5e2b493bc9368d69eeb57a69e41fa4b38ed584
  Stored in directory: /root/.cache/pip/wheels/ac/39/b3/cbd7f9cbb76501d2d5fbc84956e70d0b94e788aac87bda465e
Successfully built hnswlib


In [6]:
# build index

#!/usr/bin/env python3
"""
Build the HNSW approximate-nearest-neighbor index over trained track embeddings.

Embeddings are L2-normalized, so inner-product distance == cosine similarity.
Outputs: artifacts/index.bin + artifacts/index_meta.json

Usage:
    python3 src/build_index.py [--artifacts artifacts] [--M 32] [--ef-construction 200]
"""

import argparse
import json
import os
import time

import hnswlib
import numpy as np


def main(args_list=None):
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--artifacts", default="artifacts")
    ap.add_argument("--M", type=int, default=32)
    ap.add_argument("--ef-construction", type=int, default=200)
    ap.add_argument("--ef-search", type=int, default=100,
                    help="default query-time ef stored in index metadata")
    args = ap.parse_args(args_list)

    emb = np.load(os.path.join(args.artifacts, "embeddings.npy"))
    n, dim = emb.shape
    print(f"Building HNSW index: {n:,} x {dim} (M={args.M}, efC={args.ef_construction})")

    index = hnswlib.Index(space="ip", dim=dim)
    index.init_index(max_elements=n, ef_construction=args.ef_construction, M=args.M)
    t0 = time.time()
    chunk = 100_000
    for start in range(0, n, chunk):
        index.add_items(emb[start:start + chunk], np.arange(start, min(start + chunk, n)))
        print(f"  indexed {min(start + chunk, n):,}/{n:,} ({time.time()-t0:.0f}s)", flush=True)
    index.set_ef(args.ef_search)

    path = os.path.join(args.artifacts, "index.bin")
    index.save_index(path)
    with open(os.path.join(args.artifacts, "index_meta.json"), "w") as f:
        json.dump({"n": n, "dim": dim, "space": "ip", "M": args.M,
                   "ef_construction": args.ef_construction,
                   "ef_search": args.ef_search}, f, indent=2)
    print(f"Saved {path} ({os.path.getsize(path)/1e6:.0f} MB, {time.time()-t0:.0f}s total)")


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        main([
            "--artifacts", "/content/drive/MyDrive/soundVector/artifacts/"
        ])
    else:
        main()

Building HNSW index: 899,224 x 256 (M=32, efC=200)
  indexed 100,000/899,224 (6s)
  indexed 200,000/899,224 (13s)
  indexed 300,000/899,224 (20s)
  indexed 400,000/899,224 (27s)
  indexed 500,000/899,224 (33s)
  indexed 600,000/899,224 (40s)
  indexed 700,000/899,224 (46s)
  indexed 800,000/899,224 (52s)
  indexed 899,224/899,224 (58s)
Saved /content/drive/MyDrive/soundVector/artifacts/index.bin (1169 MB, 70s total)


In [7]:
# sanity check

#!/usr/bin/env python3
"""
Retrieval sanity check: learned embeddings vs. the raw 8D audio-feature baseline.

Metrics (higher is better), over randomly sampled anchor tracks:
  same-artist recall@50 — another track by the anchor's artist appears in top 50
  genre precision@10    — fraction of top-10 neighbors sharing >= 1 genre tag
  same-artist@10        — fraction of top-10 neighbors by the same artist
                          (context metric: too high means the space collapsed
                          to artist identity; too low means no artist signal)

The baseline reproduces the old system's space: Euclidean distance over the 8
raw audio features. Also prints a few example neighbor lists for eyeballing.

Usage:
    python3 src/sanity_check.py [--anchors 2000] [--artifacts artifacts]
"""

import argparse
import json
import os
import time

import hnswlib
import numpy as np
import pandas as pd

RAW_AUDIO_DIMS = 8  # energy..instrumentalness + tempo_norm = first 8 dense columns


def genre_sets(artifacts_dir, n_tracks):
    idx = np.load(os.path.join(artifacts_dir, "genre_indices.npy"))
    off = np.load(os.path.join(artifacts_dir, "genre_offsets.npy"))
    return [frozenset(idx[off[i]:off[i + 1]].tolist()) for i in range(n_tracks)]


def brute_force_topk(queries, corpus, k, chunk=256):
    """Euclidean top-k of each query against the full corpus (exact)."""
    out = np.zeros((len(queries), k), dtype=np.int64)
    corpus_sq = (corpus ** 2).sum(axis=1)
    for start in range(0, len(queries), chunk):
        q = queries[start:start + chunk]
        d2 = corpus_sq[None, :] - 2.0 * (q @ corpus.T) + (q ** 2).sum(axis=1)[:, None]
        part = np.argpartition(d2, k, axis=1)[:, :k + 1]
        row_d = np.take_along_axis(d2, part, axis=1)
        out[start:start + chunk] = np.take_along_axis(
            part, np.argsort(row_d, axis=1), axis=1)[:, :k]
    return out


def score(neighbors, anchors, artist_gid, gsets, k_recall=50, k_prec=10):
    """neighbors: [n_anchors, >=k_recall+1] candidate ids (may include self)."""
    recall_hits, prec_vals, same_artist_vals = [], [], []
    for row, a in zip(neighbors, anchors):
        row = row[row != a][:k_recall]
        same = artist_gid[row] == artist_gid[a]
        recall_hits.append(bool(same.any()) if artist_gid[a] >= 0 else False)
        top = row[:k_prec]
        if gsets[a]:
            prec_vals.append(np.mean([bool(gsets[a] & gsets[j]) for j in top]))
        same_artist_vals.append(np.mean(same[:k_prec]) if artist_gid[a] >= 0 else 0.0)
    return (float(np.mean(recall_hits)), float(np.mean(prec_vals)),
            float(np.mean(same_artist_vals)))


def main(args_list=None):
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--artifacts", default="artifacts")
    ap.add_argument("--anchors", type=int, default=2000)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--examples", type=int, default=3)
    args = ap.parse_args(args_list)

    rng = np.random.default_rng(args.seed)
    meta = pd.read_parquet(os.path.join(args.artifacts, "meta.parquet"))
    emb = np.load(os.path.join(args.artifacts, "embeddings.npy"))
    dense = np.load(os.path.join(args.artifacts, "features_dense.npy"))
    n = len(meta)
    artist_gid = meta["artist_gid"].to_numpy()
    print(f"{n:,} tracks | embeddings {emb.shape}")

    print("Loading genre sets...")
    gsets = genre_sets(args.artifacts, n)

    # Anchors: tracks whose artist has >= 2 tracks (so recall@50 is achievable)
    sizes = np.bincount(artist_gid[artist_gid >= 0])
    eligible = np.flatnonzero((artist_gid >= 0) & (sizes[np.clip(artist_gid, 0, None)] >= 2))
    anchors = rng.choice(eligible, size=min(args.anchors, len(eligible)), replace=False)

    with open(os.path.join(args.artifacts, "index_meta.json")) as f:
        idx_meta = json.load(f)
    index = hnswlib.Index(space=idx_meta["space"], dim=idx_meta["dim"])
    index.load_index(os.path.join(args.artifacts, "index.bin"), max_elements=idx_meta["n"])
    index.set_ef(200)

    print(f"Querying HNSW for {len(anchors)} anchors...")
    t0 = time.time()
    learned_nb, _ = index.knn_query(emb[anchors], k=51)
    ann_ms = (time.time() - t0) / len(anchors) * 1000

    print("Brute-force raw-8D baseline (exact Euclidean, this is the old system's space)...")
    t0 = time.time()
    raw = dense[:, :RAW_AUDIO_DIMS].astype(np.float32)
    baseline_nb = brute_force_topk(raw[anchors], raw, k=51)
    bf_s = time.time() - t0

    l_recall, l_prec, l_same = score(learned_nb, anchors, artist_gid, gsets)
    b_recall, b_prec, b_same = score(baseline_nb, anchors, artist_gid, gsets)

    print("\n=== Retrieval quality (n=%d anchors) ===" % len(anchors))
    print(f"{'metric':30s} {'raw 8D (old)':>14s} {'learned (new)':>14s}")
    print(f"{'same-artist recall@50':30s} {b_recall:>14.3f} {l_recall:>14.3f}")
    print(f"{'genre precision@10':30s} {b_prec:>14.3f} {l_prec:>14.3f}")
    print(f"{'same-artist share@10':30s} {b_same:>14.3f} {l_same:>14.3f}")
    print(f"\nANN query latency: {ann_ms:.2f} ms/track "
          f"(brute-force baseline took {bf_s:.0f}s for the same anchors)")

    names = meta["name"].to_numpy()
    artists = meta["artist"].to_numpy()
    pop = meta["popularity"].to_numpy()
    show = [a for a in anchors if pop[a] >= 60][:args.examples] or anchors[:args.examples].tolist()
    for a in show:
        print(f"\n--- Neighbors of: {names[a]} — {artists[a]} ---")
        row = learned_nb[list(anchors).index(a)]
        for j in row[row != a][:8]:
            print(f"    {names[j][:48]:50s} {artists[j][:28]:30s} pop {pop[j]}")

    out = {
        "n_anchors": int(len(anchors)),
        "learned": {"same_artist_recall@50": l_recall, "genre_precision@10": l_prec,
                    "same_artist_share@10": l_same, "ann_ms_per_query": round(ann_ms, 3)},
        "raw_8d_baseline": {"same_artist_recall@50": b_recall, "genre_precision@10": b_prec,
                            "same_artist_share@10": b_same},
    }
    with open(os.path.join(args.artifacts, "sanity_results.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"\nSaved {args.artifacts}/sanity_results.json")


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        main([
            "--artifacts", "/content/drive/MyDrive/soundVector/artifacts/",
            "--anchors", "2000"
        ])
    else:
        main()

899,224 tracks | embeddings (899224, 256)
Loading genre sets...
Querying HNSW for 2000 anchors...
Brute-force raw-8D baseline (exact Euclidean, this is the old system's space)...

=== Retrieval quality (n=2000 anchors) ===
metric                           raw 8D (old)  learned (new)
same-artist recall@50                   0.291          0.984
genre precision@10                      0.104          0.977
same-artist share@10                    0.038          0.777

ANN query latency: 0.05 ms/track (brute-force baseline took 17s for the same anchors)

--- Neighbors of: Can't Love Myself — Unknown Artist ---
    Sing to You                                        Unknown Artist                 pop 0
    trust u                                            Unknown Artist                 pop 0
    Anxiety Arise                                      Unknown Artist                 pop 33
    With Somebody Else                                 Unknown Artist                 pop 45
    Say Goodbye   

In [8]:
import os
import json
import numpy as np
import pandas as pd
import hnswlib

# Update path if saving directly to Google Drive
ARTIFACTS_DIR = "/content/drive/MyDrive/soundVector/artifacts/"
# ARTIFACTS_DIR = "artifacts"  # Or local path

# 1. Load Metadata & Index
meta = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "meta.parquet"))
embeddings = np.load(os.path.join(ARTIFACTS_DIR, "embeddings.npy"))

with open(os.path.join(ARTIFACTS_DIR, "index_meta.json")) as f:
    idx_meta = json.load(f)

index = hnswlib.Index(space=idx_meta["space"], dim=idx_meta["dim"])
index.load_index(os.path.join(ARTIFACTS_DIR, "index.bin"), max_elements=idx_meta["n"])
index.set_ef(100)

# 2. Pick a high-popularity seed track (e.g., top track in dataset)
popular_tracks = meta[meta["popularity"] > 80].index
seed_idx = popular_tracks[0] if len(popular_tracks) > 0 else 0

seed_name = meta.loc[seed_idx, "name"]
seed_artist = meta.loc[seed_idx, "artist"]
print(f"🎵 SEED TRACK: '{seed_name}' by {seed_artist}\n" + "="*60)

# 3. Query top-10 nearest neighbors
query_vec = embeddings[seed_idx]
neighbor_ids, distances = index.knn_query(query_vec, k=11)

print(f"{'#':<3} {'Track Title':<45} {'Artist':<25} {'Similarity':<10}")
print("-" * 85)
for rank, (nbr_id, dist) in enumerate(zip(neighbor_ids[0], distances[0])):
    if nbr_id == seed_idx:
        continue  # Skip self

    title = meta.loc[nbr_id, "name"][:44]
    artist = meta.loc[nbr_id, "artist"][:24]
    similarity = 1.0 - dist  # Cosine similarity for inner product space
    print(f"{rank:<3} {title:<45} {artist:<25} {similarity:.4f}")


🎵 SEED TRACK: 'Delicate' by Taylor Swift
#   Track Title                                   Artist                    Similarity
-------------------------------------------------------------------------------------
1   Delicate                                      Taylor Swift              0.9938
2   Dancing With Our Hands Tied                   Taylor Swift              0.9930
3   I Did Something Bad                           Taylor Swift              0.9912
4   Don’t Blame Me                                Taylor Swift              0.9901
5   ...Ready For It?                              Taylor Swift              0.9898
6   Getaway Car                                   Taylor Swift              0.9882
7   Miss Americana & The Heartbreak Prince        Taylor Swift              0.9877
8   So It Goes...                                 Taylor Swift              0.9873
9   ...Ready For It?                              Taylor Swift              0.9873
10  Getaway Car                        

In [9]:
import os
import json
import numpy as np
import pandas as pd
import hnswlib

# Set path to your artifacts directory
ARTIFACTS_DIR = "/content/drive/MyDrive/soundVector/artifacts/"
# ARTIFACTS_DIR = "artifacts"  # Use this if running locally in workspace

# ── 1. Load Catalog & Vector Index ─────────────────────────────────────────
print("⏳ Loading metadata and vector index...")
meta = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "meta.parquet"))
embeddings = np.load(os.path.join(ARTIFACTS_DIR, "embeddings.npy"))

with open(os.path.join(ARTIFACTS_DIR, "index_meta.json")) as f:
    idx_meta = json.load(f)

index = hnswlib.Index(space=idx_meta["space"], dim=idx_meta["dim"])
index.load_index(os.path.join(ARTIFACTS_DIR, "index.bin"), max_elements=idx_meta["n"])
index.set_ef(100)
print(f"✅ Ready! Searching across {len(meta):,} tracks.")


# ── 2. Recommendation Engine Function ─────────────────────────────────────
def recommend(song_title: str, top_k: int = 10):
    """Search for a track title and print vector similarity recommendations."""
    query_lower = song_title.lower().strip()

    # Search for matching tracks by title (case-insensitive sub-string match)
    matches = meta[meta["name"].str.lower().str.contains(query_lower, na=False, regex=False)]

    if len(matches) == 0:
        print(f"❌ No tracks found matching '{song_title}'. Try another search!")
        return

    # Sort matches by popularity to find the most likely intended track
    matches = matches.sort_values(by="popularity", ascending=False)

    # Pick the top match (highest popularity)
    seed_idx = matches.index[0]
    seed_track = meta.loc[seed_idx]

    print(f"\n" + "="*80)
    print(f"🎯 SEED TRACK : {seed_track['name']} — {seed_track['artist']}")
    print(f"   (Year: {seed_track['release_year']}, Popularity: {seed_track['popularity']}/100)")
    if len(matches) > 1:
        print(f"   ℹ️  (Found {len(matches)} total matches; selected most popular match)")
    print("="*80 + "\n")

    # Perform HNSW Vector Query (get k+1 to exclude the seed track itself)
    seed_vec = embeddings[seed_idx]
    nbr_ids, distances = index.knn_query(seed_vec, k=top_k + 1)

    print(f"{'#':<3} {'Similarity':<12} {'Track Title':<42} {'Artist':<22} {'Year':<6}")
    print("-" * 88)

    rank = 1
    for nbr_id, dist in zip(nbr_ids[0], distances[0]):
        if nbr_id == seed_idx:
            continue  # Skip self

        sim_score = (1.0 - dist) * 100  # Inner Product similarity %
        title = str(meta.loc[nbr_id, "name"])[:40]
        artist = str(meta.loc[nbr_id, "artist"])[:20]
        year = meta.loc[nbr_id, "release_year"]

        print(f"{rank:<3} {sim_score:>6.1f}%       {title:<42} {artist:<22} {year:<6}")
        rank += 1
        if rank > top_k:
            break
    print("-" * 88 + "\n")


# ── 3. Interactive Input Loop ──────────────────────────────────────────────
# Try a few default examples first:
recommend("Blinding Lights")
recommend("Shape of You")

while True:
    user_input = input("Enter track name (or 'q' to exit): ")
    if user_input.lower() in ("q", "quit", "exit"):
        break
    recommend(user_input)


⏳ Loading metadata and vector index...
✅ Ready! Searching across 899,224 tracks.

🎯 SEED TRACK : Blinding Lights — The Weeknd
   (Year: 2020, Popularity: 90/100)
   ℹ️  (Found 97 total matches; selected most popular match)

#   Similarity   Track Title                                Artist                 Year  
----------------------------------------------------------------------------------------
1     99.6%       Havana (feat. Young Thug)                  Camila Cabello         2018  
2     74.6%       Havana - Remix                             Camila Cabello         2017  
3     72.0%       Call Out My Name                           The Weeknd             2018  
4     69.4%       Blinding Lights                            The Weeknd             2020  
5     64.1%       Never Be the Same                          Camila Cabello         2018  
6     63.7%       She Loves Control                          Camila Cabello         2018  
7     62.8%       Shameless                        

In [10]:
import os
import json
import numpy as np
import pandas as pd
import hnswlib

# Set path to your artifacts directory
ARTIFACTS_DIR = "/content/drive/MyDrive/soundVector/artifacts/"
# ARTIFACTS_DIR = "artifacts"  # Use this if running locally in workspace

# ── 1. Load Catalog & Vector Index ─────────────────────────────────────────
print("⏳ Loading metadata and vector index...")
meta = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "meta.parquet"))
embeddings = np.load(os.path.join(ARTIFACTS_DIR, "embeddings.npy"))

with open(os.path.join(ARTIFACTS_DIR, "index_meta.json")) as f:
    idx_meta = json.load(f)

index = hnswlib.Index(space=idx_meta["space"], dim=idx_meta["dim"])
index.load_index(os.path.join(ARTIFACTS_DIR, "index.bin"), max_elements=idx_meta["n"])
index.set_ef(100)
print(f"✅ Ready! Searching across {len(meta):,} tracks.")


# ── 2. Recommendation Engine Function ─────────────────────────────────────
def recommend(song_title: str, top_k: int = 25):
    """Search for a track title and print vector similarity recommendations."""
    query_lower = song_title.lower().strip()

    # Search for matching tracks by title (case-insensitive sub-string match)
    matches = meta[meta["name"].str.lower().str.contains(query_lower, na=False, regex=False)]

    if len(matches) == 0:
        print(f"❌ No tracks found matching '{song_title}'. Try another search!")
        return

    # Sort matches by popularity to find the most likely intended track
    matches = matches.sort_values(by="popularity", ascending=False)

    # Pick the top match (highest popularity)
    seed_idx = matches.index[0]
    seed_track = meta.loc[seed_idx]

    print(f"\n" + "="*80)
    print(f"🎯 SEED TRACK : {seed_track['name']} — {seed_track['artist']}")
    print(f"   (Year: {seed_track['release_year']}, Popularity: {seed_track['popularity']}/100)")
    if len(matches) > 1:
        print(f"   ℹ️  (Found {len(matches)} total matches; selected most popular match)")
    print("="*80 + "\n")

    # Perform HNSW Vector Query (get k+1 to exclude the seed track itself)
    seed_vec = embeddings[seed_idx]
    nbr_ids, distances = index.knn_query(seed_vec, k=top_k + 1)

    print(f"{'#':<3} {'Similarity':<12} {'Track Title':<42} {'Artist':<22} {'Year':<6}")
    print("-" * 88)

    rank = 1
    for nbr_id, dist in zip(nbr_ids[0], distances[0]):
        if nbr_id == seed_idx:
            continue  # Skip self

        sim_score = (1.0 - dist) * 100  # Inner Product similarity %
        title = str(meta.loc[nbr_id, "name"])[:40]
        artist = str(meta.loc[nbr_id, "artist"])[:20]
        year = meta.loc[nbr_id, "release_year"]

        print(f"{rank:<3} {sim_score:>6.1f}%       {title:<42} {artist:<22} {year:<6}")
        rank += 1
        if rank > top_k:
            break
    print("-" * 88 + "\n")


# ── 3. Interactive Input Loop ──────────────────────────────────────────────
# Try a few default examples first:
recommend("Blinding Lights")
recommend("Shape of You")
recommend("There's nothing holding me back")
recommend("badtameez dil")

# while True:
#     user_input = input("Enter track name (or 'q' to exit): ")
#     if user_input.lower() in ("q", "quit", "exit"):
#         break
#     recommend(user_input)


⏳ Loading metadata and vector index...
✅ Ready! Searching across 899,224 tracks.

🎯 SEED TRACK : Blinding Lights — The Weeknd
   (Year: 2020, Popularity: 90/100)
   ℹ️  (Found 97 total matches; selected most popular match)

#   Similarity   Track Title                                Artist                 Year  
----------------------------------------------------------------------------------------
1     99.6%       Havana (feat. Young Thug)                  Camila Cabello         2018  
2     74.6%       Havana - Remix                             Camila Cabello         2017  
3     72.0%       Call Out My Name                           The Weeknd             2018  
4     69.4%       Blinding Lights                            The Weeknd             2020  
5     64.1%       Never Be the Same                          Camila Cabello         2018  
6     63.7%       She Loves Control                          Camila Cabello         2018  
7     62.8%       Shameless                        